In [39]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# load Dataset


In [40]:
data = pd.read_csv('IMDB Dataset.csv')

data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [41]:
data.shape

(50000, 2)

In [42]:
data.tail()

,review,sentiment
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


In [43]:
data["sentiment"].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

# one hot encoding

In [44]:
data.replace({"sentiment": {"positive":1, "negative": 0}}, inplace=True)

# data preprocessing

In [45]:
import os
import tensorflow as tf

from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [46]:
train_data, test_data = train_test_split(data, test_size = 0.2, random_state = 42)

In [47]:
train_data.shape

(40000, 2)

In [48]:
test_data.shape

(10000, 2)

In [49]:
# Configure TensorFlow to use GPU with mixed precision if available
physical_devices = tf.config.list_physical_devices('GPU')
gpu_available = bool(physical_devices)

if gpu_available:
    try:
        # Enable memory growth for all GPUs
        for gpu in physical_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
        
        # Set mixed precision policy for GPUs
        # Using 'mixed_float16' for computation, 'float32' for variables
        policy = tf.keras.mixed_precision.Policy('mixed_float16')
        tf.keras.mixed_precision.set_global_policy(policy)
        
    except RuntimeError as e:
        gpu_available = False

if not gpu_available:
    # Fallback to CPU with float32 policy
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # Force CPU
    policy = tf.keras.mixed_precision.Policy('float32')
    tf.keras.mixed_precision.set_global_policy(policy)


In [50]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_data["review"])

In [51]:
x_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]),maxlen=200)
x_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]),maxlen=200)

In [52]:
x_train

array([[1935,    1, 1200, ...,  205,  351, 3856],
       [   3, 1651,  595, ...,   89,  103,    9],
       [   0,    0,    0, ...,    2,  710,   62],
       ...,
       [   0,    0,    0, ..., 1641,    2,  603],
       [   0,    0,    0, ...,  245,  103,  125],
       [   0,    0,    0, ...,   70,   73, 2062]],
      shape=(40000, 200), dtype=int32)

In [53]:
x_test

array([[   0,    0,    0, ...,  995,  719,  155],
       [  12,  162,   59, ...,  380,    7,    7],
       [   0,    0,    0, ...,   50, 1088,   96],
       ...,
       [   0,    0,    0, ...,  125,  200, 3241],
       [   0,    0,    0, ..., 1066,    1, 2305],
       [   0,    0,    0, ...,    1,  332,   27]],
      shape=(10000, 200), dtype=int32)

In [54]:
y_train = train_data["sentiment"]
y_test = test_data["sentiment"]

In [55]:
y_train

39087    0
30893    0
45278    1
16398    0
13653    0
        ..
11284    1
44732    1
38158    0
860      1
15795    1
Name: sentiment, Length: 40000, dtype: int64

In [56]:
model = Sequential()
model.add(Embedding(input_dim=5000,output_dim=128,input_length=200))
model.add(LSTM(128, dropout=0.2,recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))


In [57]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [58]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [59]:
model.fit(x_train,y_train,epochs=5,batch_size=64, validation_split= 0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 109s 213ms/step - accuracy: 0.7880 - loss: 0.4565 - val_accuracy: 0.8470 - val_loss: 0.3628
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 113s 227ms/step - accuracy: 0.8377 - loss: 0.3766 - val_accuracy: 0.8594 - val_loss: 0.3463
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 115s 229ms/step - accuracy: 0.8782 - loss: 0.3011 - val_accuracy: 0.8449 - val_loss: 0.3594
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 115s 229ms/step - accuracy: 0.8856 - loss: 0.2796 - val_accuracy: 0.8579 - val_loss: 0.3620
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 113s 227ms/step - accuracy: 0.9061 - loss: 0.2379 - val_accuracy: 0.8679 - val_loss: 0.3209


In [60]:
loss, accuracy = model.evaluate(x_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 34ms/step - accuracy: 0.8803 - loss: 0.3054


In [61]:
print(loss)

0.3054279685020447


In [62]:
print(accuracy)

0.880299985408783


# building predictive system

In [63]:
def predictive_system(review):
    sequences = tokenizer.texts_to_sequences([review])
    Padded_sequences = pad_sequences(sequences, maxlen=200)
    prediction = model.predict(Padded_sequences)
    sentiment= "positive" if prediction[0][0] > 0.5 else "negative"
    return sentiment

In [64]:
predictive_system("this is a good movie")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 259ms/step


'positive'

In [65]:
predictive_system("this is bad ")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step


'negative'

In [66]:
model.save("model.h5")

In [67]:
import joblib
joblib.dump(tokenizer, "tokenizer.joblib")

['tokenizer.joblib']